# ML1 - GridSearch with DecisionTreeRegressor, RandomForestRegressor

This notebook follows the ML1_Pipeline_Model notebook and focuses on:
1. Using DecisionTreeRegressor and RandomForestRegressor (or others)  
2. Hyperparameter tuning with GridSearchCV  
3. Cross-validation

## 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso 
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier, export_text
from scipy.stats import uniform
from scipy.stats import randint


## 2. Loading and Preparing Data with the Pipeline

We follow the same preparation steps as in the previous notebook.


In [ ]:
# Load the data from the pickle file.
df = pd.read_pickle('arbres_clean_Laura.pkl')
#Prepare the train and test sets
X = df.drop(columns=['anneedeplantation'])
y = df['anneedeplantation']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=20)

categorical_variables = X_train.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

numerical_variables = X_train.select_dtypes(include = ['number']).columns.tolist()
print(categorical_variables,numerical_variables)

#pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, categorical_variables),
    ('num', numerical_pipeline, numerical_variables)])
	
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
#permet de savoir combien de variables on a après le pipeline, ici 421 colonnes
X_train_transformed.shape

## 3. Initial Cross-Validation

Before tuning hyperparameters, let's evaluate a model with default parameters using cross-validation.


In [ ]:
# Initial cross-validation without hyperparameter tuning  
# We can also print the MSE or MAE scores from each cross-validation fold.

model_linear = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', LinearRegression())
])
model_lasso = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', Lasso(alpha=0.001, max_iter=10000))
])

#max_depth est plus ou moins nb de variables explorées
model_random_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', RandomForestRegressor(max_depth=30, min_samples_leaf=0.05,random_state=42))
])

model_random_forest_RS = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', RandomForestRegressor(max_depth=40, min_samples_leaf=0.05,
                                       n_estimators=64, random_state=42))
])

model_decision_tree = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', DecisionTreeClassifier(max_depth=10, random_state=42))
])

model_gradient_boosting_regressor = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', GradientBoostingRegressor(random_state=42))
])




In [ ]:
#application avec la régression linéaire de base
arbres_scores = cross_val_score(model_linear, X_train, y_train, cv = 5, scoring='neg_mean_squared_error') 

print("MSE moyen :", np.mean(-arbres_scores))
print("Écart-type :", np.std(-arbres_scores))

## 4. 🌳 Modeling with GridSearchCV

Let's choose two machine learning models.

Evaluate the models with hyperparameter tuning (testing different hyperparameter values).  
Optimize the models as you wish, for example for Decision Tree and Random Forest:

### 🔍 Hyperparameters Tested
- `max_depth`: maximum depth of the trees [3, 4, 5, 10]  
- `min_samples_leaf`: minimum number of samples required at a leaf [0.1, 0.05]  
- `n_estimators` (for Random Forest only): number of trees in the forest [50, 100, 200]

Let's use GridSearchCV to automate the search for the best hyperparameters.

In [ ]:
# Hyperparameter tuning with GridSearchCV, and we can print the MSE or MAE scores for each cross-validation fold.

param_grid = [
 {'regression__alpha': [0.001, 0.01, 0.1, 1, 3, 10, 100, 1000]}
 ]

#arbres_scores_lasso = cross_val_score(model_lasso, X_train, y_train, cv = 5, scoring='neg_mean_squared_error')
arbres_grid_search = GridSearchCV(model_lasso, param_grid, cv=5, scoring='neg_mean_squared_error')
arbres_grid_search.fit(X_train, y_train)

arbres_grid_results = pd.DataFrame(arbres_grid_search.cv_results_)

print(arbres_grid_results)
print(arbres_grid_search.best_estimator_)



In [ ]:
#GridSearch avec la random forest
param_grid_forest = {
    'regression__max_depth': [400],
    'regression__min_samples_leaf': [0.05, 0.01, 0.001, 0.0001],
    'regression__n_estimators': [250, 300, 350, 400]
}

arbres_RF_grid_search = GridSearchCV(model_random_forest, param_grid_forest, cv=5, scoring='neg_mean_squared_error')
arbres_RF_grid_search.fit(X_train, y_train)

arbres_RF_grid_results = pd.DataFrame(arbres_RF_grid_search.cv_results_)

In [ ]:
print(arbres_RF_grid_results)
print(arbres_RF_grid_search.best_estimator_)

In [ ]:
#Plutôt que d'utiliser un gridsearch, plutôt utiliser un randomsearch, plus rapide


#RandomizedSearchCV avec la random forest

distributions = {
    'regression__max_depth': [3, 4, 5, 10, None],
    'regression__min_samples_leaf': [0.05, 0.1],
    'regression__n_estimators': randint(50, 201)
}

arbres_RF_random_search = RandomizedSearchCV(
    model_random_forest_RS,
    param_distributions=distributions,
    n_iter=10,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

arbres_RF_random_search.fit(X_train, y_train)

arbres_RF_random_results = pd.DataFrame(arbres_RF_random_search.cv_results_)

print(arbres_RF_random_results)
print(arbres_RF_random_search.best_estimator_)


In [ ]:
#RandomizedSearchCV avec le gradient boosting

distributions = {
    'regression__max_depth': [3, 4, 5, 10, None],
    'regression__min_samples_leaf': [0.05, 0.1],
    'regression__n_estimators': randint(50, 201)
}

arbres_RF_random_search = RandomizedSearchCV(
    model_random_forest_RS,
    param_distributions=distributions,
    n_iter=10,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

arbres_RF_random_search.fit(X_train, y_train)

arbres_RF_random_results = pd.DataFrame(arbres_RF_random_search.cv_results_)

print(arbres_RF_random_results)
print(arbres_RF_random_search.best_estimator_)


In [ ]:
# Code for evaluating the best model on the test set.

# Créer un modèle de référence
dummy = DummyRegressor(strategy='mean')

# L'entraîner sur les données d'entraînement
dummy.fit(X_train_transformed, y_train)

# Prédire les années sur le test
y_pred_dummy = dummy.predict(X_test)

dummy_scores = cross_val_score(
    dummy,
    X_train_transformed,
    y_train,
    cv=10,
    scoring='neg_mean_squared_error'
)



## 5. Final Evaluation on the Test Set

In [ ]:
from sklearn.model_selection import cross_validate
import numpy as np

scoring = {
    'mse': 'neg_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}

# modèles Pipeline
models = {
    'LinearRegression': model_linear,
    'Lasso': model_lasso,
    'Dummy': dummy,
    'Random Forest' : model_random_forest_RS,
    'Decision Tree' : model_decision_tree,
    'Gradient Boosting' : model_gradient_boosting_regressor 
}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring=scoring
    )

    mse = -scores['test_mse']
    mae = -scores['test_mae']
    r2 = scores['test_r2']

    rmse = np.sqrt(mse)

    print(f"\n{name}")
    print(f"MSE moyen  : {mse.mean():.2f}")
    print(f"RMSE moyen : {rmse.mean():.2f}")
    print(f"MAE moyen  : {mae.mean():.2f}")
    print(f"R² moyen   : {r2.mean():.3f}")
    print(f"Écart-type MAE : {mae.std():.2f}")

## 6. Visualization of Predictions  
One model will suffice.

In [ ]:
# Visualisation of predictions.

## [OPTIONAL] 7.Feature Importance

Using the random forest, how can we determine which features are the most important?

In [ ]:
# Code. In your opinion, how is feature importance calculated in Random Forest?

Typically, one might consider using backward elimination on feature importance to select the best subset of features, but this can be time-consuming if there are many features and the model takes a while to train, as is the case here. Instead, we can use Lasso-regularized regression to eliminate features, before retraining a RandomForest on the selected features. Create a pipeline with `LassoCV` as the selector and a RandomForest as the model with the best hyperparameters found by the previous grid-search. Evaluate on the test set.

In [ ]:
# Code

## 8. [OPTIONAL]  Compare more different models

Compare more regression models with one another (for example, in addition to linear regression, decision trees, and forests, use `GradientBoostingRegressor`, and the novel `TabICLRegressor` if you have time ( https://github.com/soda-inria/tabicl ))), by performing appropriate hyperparameter tuning for each model:

In [ ]:
# Code. Search which hyper-parameters are worth tuning for each model

Choose the model that performs best and its optimal hyperparameters, then select the best subset of features for this model, and finally perform one last grid search on your model’s hyperparameters using the selected features to obtain your final model. Evaluate it on the test set.

In [ ]:
# Code

## Conclusion